# Monetary data extraction from Arabic MIUs (Batch pipeline)

This notebook runs a **manifest-driven, resumable** OpenAI Batch pipeline to extract structured monetary information from Arabic MIUs. It is the **archival, reproducible** version of the workflow used in the dissertation:

> **Hamid Reza Hakimi**, *Monetary Equivalents in Premodern Islamic Historical and Biographical Texts (1–1000 AH/600–1600 CE): Algorithmic Analysis into Economic History* (PhD dissertation, University of Hamburg).

It implements the pipeline described in **Chapter 1, §1.2.2** for extracting structured monetary information from Arabic Minimum Information Units (MIUs) using **GPT-5.1** via the OpenAI **Batch API**.


## What you get
- Deterministic chunking into JSONL input parts (`batch_work/inputs/batch_part_XXXX.jsonl`)
- Batch submission + polling + download, tracked in `batch_work/manifest.json`
- Robust parsing of Batch outputs (JSON schema validation)
- **Streaming merge** suitable for very large corpora (100k+ MIUs)
- **Automatic retries** for failed/cancelled/expired chunks (configurable attempts + backoff)

## Typical workflow
Run cells in order:
1. Install/imports
2. Configure paths + chunking + retry policy
3. Load corpus + few-shot
4. Build chunks + manifest
5. Submit pending chunks
6. Poll statuses (run periodically until most chunks are completed)
7. Retry failed chunks (optional; safe to re-run)
8. Download outputs
9. Parse + merge + export artifacts

## Files written
All artifacts live under `batch_work/`:
- `inputs/`  chunked JSONL requests
- `outputs/` per-chunk Batch outputs
- `manifest.json` pipeline state (safe to commit for reproducibility)
- `merged_miu.json` merged corpus with `ann.monetary_info`
- `errors.json` parsing/API errors keyed by `miu_id`
- `empty_hits.json` MIUs where the model returned an empty `monetary_info` list (optional)

> Tip for large corpora: keep chunk sizes modest (e.g., 1k–5k MIUs). Retries become cheaper, and downloads/merges are faster.


In [ ]:
# Install dependencies (Colab-friendly). If you're running locally, you can skip this cell.
!pip -q install -U openai jsonschema


In [ ]:
import json
import os
import re
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

from openai import OpenAI
from jsonschema import validate as jsonschema_validate
from jsonschema.exceptions import ValidationError


In [ ]:
# OpenAI client (expects OPENAI_API_KEY in your environment)
# Colab users: set it via Secrets or os.environ["OPENAI_API_KEY"] = "..."
def get_client() -> OpenAI:
    """Create and return an authenticated OpenAI client.

    Args:
        None

    Returns:
        OpenAI: Configured client instance using the OPENAI_API_KEY environment variable.

    Raises:
        RuntimeError: If OPENAI_API_KEY is not set.
    """
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set in the environment.")
    return OpenAI(api_key=api_key)

client = get_client()

In [ ]:
# -----------------------
# Configuration (edit me)
# -----------------------

MIU_JSON_PATH = Path("sample_MIU_corpus.json")   # mapping: miu_id -> {"txt": ...}
FEW_SHOT_PATH = Path("few_shot_examples.json")                    # list[{text, output}]

WORK_DIR = Path("batch_work")   # all generated artifacts live here
INPUT_DIR = WORK_DIR / "inputs"
OUTPUT_DIR = WORK_DIR / "outputs"

MODEL = "gpt-5.1"
COMPLETION_WINDOW = "24h"       # e.g., "24h"
POLL_SECONDS = 15

# Chunking strategy (deterministic):
# - max_records keeps requests manageable and makes retries cheap.
# - max_bytes prevents overly-large JSONL parts (Batch API file limits).
MAX_RECORDS_PER_CHUNK = 2000
MAX_BYTES_PER_CHUNK = 90 * 1024 * 1024  # 90MB (leave headroom under 100MB)

# Merge behavior
REQUIRE_NONEMPTY = True  # if True: don't merge empty monetary_info; record them as "empty_hits"
SKIP_ALREADY_ANNOTATED = True  # if True: skip MIUs that already have ann.monetary_info in the input corpus

# Retry policy for large runs
RETRY_ON_STATUSES = ("failed", "cancelled", "expired")
MAX_RETRY_ATTEMPTS = 3
RETRY_BACKOFF_BASE_SECONDS = 120   # exponential backoff base (2 minutes)
RETRY_BACKOFF_MAX_SECONDS = 3600   # cap backoff at 1 hour

# Large-corpus merge strategy
# If True, merge outputs into miu_data incrementally (low memory).
STREAMING_MERGE = True



In [ ]:
# -----------------------
# Load corpus + few-shot
# -----------------------

miu_data: Dict[str, Dict[str, Any]] = json.loads(MIU_JSON_PATH.read_text(encoding="utf-8"))
few_shot: List[Dict[str, Any]] = json.loads(FEW_SHOT_PATH.read_text(encoding="utf-8"))

print(f"Loaded MIUs: {len(miu_data):,}")
print(f"Loaded few-shot examples: {len(few_shot):,}")

# Quick sanity check
sample_id = next(iter(sorted(miu_data.keys())))
print("Sample miu_id:", sample_id)
print("Sample text (first 200 chars):", (miu_data[sample_id].get("txt") or "")[:200])


Loaded MIUs: 10
Loaded few-shot examples: 3
Sample miu_id: 0306Wakic.AkhbarQudat.845940806488
Sample text (first 200 chars): مملوكا جارية أو غلاما وكرهه فرده بعيب فقال له البائع : لا ترده فأنا اربح لك فيه دنانير . قال : أو تفعل ؟ قال : نعم . قال فكرهه وهب ولم يعرضه . فدعى به شريك فقال : ألم تقل إنك تربحه فيه ؟ قال : بلى قد 


In [ ]:
# Structured Outputs JSON Schema for EACH record
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "name": "MonetaryInfoResponse",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "monetary_info": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "exact_phrase_in_text_Arabic": {"type": "string"},
                        "amount_of_money_in_digits": {"type": "string"},
                        "currency_Arabic": {"type": "string"},
                        "currency": {"type": "string"},
                        "year_of_transaction_Arabic": {"type": "string"},
                        "year_of_transaction_in_digits": {"type": "string"},
                        "other_years_Arabic": {"type": "array", "items": {"type": "string"}},
                        "other_years_in_digits": {"type": "array", "items": {"type": "string"}},
                        "location_of_transaction_Arabic": {"type": "string"},
                        "amount_of_item_in_transaction_Arabic": {"type": "string"},
                        "payer_or_owner_Arabic": {"type": "string"},
                        "payer_or_owner_categories_controlled": {"type": "array", "items": {"type": "string"}},
                        "payer_or_owner_categories_free": {"type": "array", "items": {"type": "string"}},
                        "payee_Arabic": {"type": "string"},
                        "payee_categories_controlled": {"type": "array", "items": {"type": "string"}},
                        "payee_categories_free": {"type": "array", "items": {"type": "string"}},
                        "money_paid_for_Arabic": {"type": "string"},
                        "reason_of_transaction": {"type": "string"},
                        "reason_categories_controlled": {"type": "array", "items": {"type": "string"}},
                        "reason_categories_free": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": [
                        "exact_phrase_in_text_Arabic",
                        "amount_of_money_in_digits",
                        "currency_Arabic",
                        "currency",
                        "year_of_transaction_Arabic",
                        "year_of_transaction_in_digits",
                        "other_years_Arabic",
                        "other_years_in_digits",
                        "location_of_transaction_Arabic",
                        "amount_of_item_in_transaction_Arabic",
                        "payer_or_owner_Arabic",
                        "payer_or_owner_categories_controlled",
                        "payer_or_owner_categories_free",
                        "payee_Arabic",
                        "payee_categories_controlled",
                        "payee_categories_free",
                        "money_paid_for_Arabic",
                        "reason_of_transaction",
                        "reason_categories_controlled",
                        "reason_categories_free",
                    ],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["monetary_info"],
        "additionalProperties": False,
    },
}

SYSTEM_RULES = """You are an expert in extracting structured monetary data from classical Arabic texts.
- Return an object with key "monetary_info": [...]
- Use ONLY phrases present in the text for exact_phrase_in_text_Arabic.
- If something is missing, output "null".
- Categories must be relevant and consistent.


GENERAL INSTRUCTIONS
1. Output format: Always return a list of JSON objects, one per occurrence of monetary information.
2. Extraction rules:
  - Catch every mention of monetary units (Dinar, Dirham, Fals, etc.) as a separate object.
  - Stick to the text for extracting information. Extract only if the evidence is explicitly present in the provided text.
    Do not make up answers, search in other sources, or guess. If information is missing, or you do not know the answer, output "null".
  - Fields ending in "_Arabic": copy the exact Arabic text (no edits).
  - Fields ending in "_in_digits": use English digits as strings.
  - Fields ending in "_controlled": select only from the relevant CONTROLLED LISTs below.
  - Fields ending in "_free": propose items irregardless of the CONTROLLED LISTs.
  - All other fields: use English in lowercase.
3. Field definitions:
  - exact_phrase_in_text_Arabic: Repeat the full Arabic sentence in the text containing the monetary information in verbatim (no paraphrase, no added/removed punctuation or diacritics not present in the source).
  - amount_of_money_in_digits: Numeric value of the amount of money as string.
  - currency_Arabic: Currency name in Arabic, including adjectives (e.g., "درهم ناصرية").
  - currency: Currency name in English.
  - year_of_transaction_Arabic: Year of transaction in Arabic.
  - year_of_transaction_in_digits: Numeric value of the year of transaction as string.
  - other_years_Arabic: List of all other years mentioned in the text in Arabic.
  - other_years_in_digits: List of numeric values of all other years mentioned in the text as string.
  - location_of_transaction_Arabic: The city in which the transaction has taken place. If the city is not mentioned, the name of the country or the historical region (like Sham) is also fine.
  - amount_of_item_in_transaction_Arabic: Amount of item in transaction.
  - payer_or_owner_Arabic: Name of the payer or the owner of the money.
  - payer_or_owner_categories_controlled: List of categories for the job, role, or profession of the payer/owner of the money, assigned from the CONTROLLED LIST below.
  - payer_or_owner_categories_free: Propose a list containing one to three categories for the job, role, or profession of the payer/owner of the money, irregardless of the CONTROLLED LIST.
  - payee_Arabic: Name of the payee.
  - payee_categories_controlled: List of categories for the job, role, or profession of the payee, assigned from the CONTROLLED LIST below.
  - payee_categories_free: Propose a list containing one to three categories for the job, role, or profession of the payee, irregardless of the CONTROLLED LIST.
  - money_paid_for_Arabic: Reason of payment copied from the text.
  - reason_of_transaction: Purpose of the monetary transaction.
  - reason_categories_controlled: List of categories for the reason of transaction, assigned from the CONTROLLED LIST below.
  - reason_categories_free: Propose a list containing one to three categories for the reason of transaction, irregardless of the CONTROLLED LIST.

CATEGORIES

Payer/Owner and payee Roles
- payer_or_owner__categories and payee_categories: job, role, or profession of the payer/owner and payee.
- CONTROLLED LIST:
  - scholar → all types of scholars (religious and non-religious)
  - companion → anyone connected to the Prophet
  - governor → provincial/state rulers, not caliphs or kings
  - judge → all legal judges (qāḍī, etc.)
  - administrative_official → viziers, scribes, tax collectors, bureaucrats
  - public_service_worker → non-military staff providing state services
  - military_personnel → soldiers, warriors, commanders
  - merchant → traders of any scale
  - ascetic → renunciants, mystics living without wealth, Sufis
  - captive → prisoners of war or hostages
  - performer → performers of music/entertainment
  - medical_staff → physicians, healers, surgeons
  - poor → all types of poor people
  - caliph, king, farmer, shopkeeper, poet, slave, scientist → as commonly understood
  - other: use only if none of the above fits.



Reasons for Transaction
- reason_categories: purpose of the monetary transaction.
- CONTROLLED LIST:
  - debt → borrowed/lent money, loans, deposits
  - remuneration → salaries, allowances, rewards, gifts
  - earnings → income from selling, renting, trading
  - wealth → personal assets or stored wealth
  - tribute → money given by one municipal leader or governor to another as submission
  - tax → all formal taxes, including zakat and jizya
  - property → houses, land (including iqta), estates
  - bribe → illicit payments
  - theft → looting, extortion, booty
  - military_expenses → costs tied to war, like weapon or sending troops
  - education_expenses → costs paid to teachers and students for teaching and learning
  - medical_expenses → costs paid for medial services
  - agricultural_expenses → costs tied to agriculture, like gardening
  - living_expenses → the amount of money a person needs to spend a day/month/year
  - party_expenses → costs of a party or a ceremony
  - position → jobs or appointments purchased/assigned
  - charity → voluntary giving, donations, sadaqa
  - waqf → endowments specifically designated as waqf
  - exchange_rate → mention of currency exchange value
  - performance → payment for musical, theatrical, or artistic performances
  - luxury → jewelry, fine goods, expensive items
  - book → manuscripts, copied works, literary purchases
  - treasury → money inside or flowed inside/outside of the treasury
  - settlement → political, legal, or other types of settlements between two individuals/groups
  - price → mention of the price of an item (when this label is used, another label identifying the item whose price is mentioned should also be included)
  - food, animal_feed, grain, animal, clothing, furniture, sweets, oil, dairy, liquid, fruit, dried_fruit, meat, fabric, slave, dowry, travel, transport,
    construction, confiscation, public_service, poetry, trade, inheritance, ransom, penalty → straightforward items
  - other: use only if none of the above fits.

CONSISTENCY RULES
- Always use lowercase for labels.
- Use snake case for compound labels (like military_personnel)
- Always choose the most precise categories possible.
- Maintain strict consistency across all extracted JSON objects.
- Each JSON object must be independent, complete, and schema-compliant.
"""



In [ ]:
# -----------------------
# Helpers (manifest-driven)
# -----------------------
#
# Design goals (esp. for 100k+ MIUs):
# - Deterministic chunking (stable across re-runs)
# - Low memory usage (stream JSONL; streaming merge)
# - Resumability via manifest.json
# - Automatic retries for failed/cancelled/expired chunks
#
# Most functions below are "safe to re-run".

import random

def sha256_text(s: str) -> str:
    """Compute a SHA-256 hex digest for a given text string.

    Args:
        s (str): Input text.

    Returns:
        str: SHA-256 digest as a lowercase hexadecimal string.
    """
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def ensure_dirs() -> None:
    """Create required working directories (inputs/outputs/logs) if missing.

    Args:
        None

    Returns:
        None
    """
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def utc_now_iso() -> str:
    """Return the current UTC time as an ISO-8601 string.

    Args:
        None

    Returns:
        str: Timestamp formatted like 'YYYY-MM-DDTHH:MM:SSZ'.
    """
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def iter_mius_sorted(data: Dict[str, Dict[str, Any]]) -> Iterable[Tuple[str, Dict[str, Any]]]:
    """Iterate over MIUs in deterministic (sorted) order.

    Args:
        data (dict): Mapping of miu_id to MIU record.

    Yields:
        tuple[str, dict]: (miu_id, record) pairs in sorted miu_id order.
    """
    for miu_id in sorted(data.keys()):
        yield miu_id, data[miu_id]

def build_messages(miu_text: str, few_shot: List[Dict[str, Any]], system_rules: str) -> List[Dict[str, str]]:
    """Build the chat message list for a single MIU extraction request.

    Args:
        miu_text (str): The Arabic MIU text to analyze.
        few_shot (list[dict]): Few-shot examples/messages to prepend (already in chat format).
        system_rules (str): System prompt containing extraction rules.

    Returns:
        list[dict]: Messages compatible with the Chat Completions/Responses API.
    """
    msgs: List[Dict[str, str]] = [{"role": "system", "content": system_rules}]
    for ex in few_shot:
        msgs.append({"role": "user", "content": f"Arabic text:\n{ex['text']}"})
        msgs.append({"role": "assistant", "content": json.dumps(ex["output"], ensure_ascii=False, indent=2)})
    msgs.append({"role": "user", "content": f"Arabic text:\n{miu_text}"})
    return msgs

def batch_record(miu_id: str, miu_text: str) -> Dict[str, Any]:
    """Create one Batch API request record for a single MIU.

    Args:
        miu_id (str): Unique MIU identifier used as custom_id for later merge.
        miu_text (str): MIU text to analyze.

    Returns:
        dict: A JSON-serializable object representing one JSONL line for batch input.
    """
    return {
        "custom_id": miu_id,
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "input": build_messages(miu_text, few_shot, SYSTEM_RULES),
            "text": {
                "format": RESPONSE_SCHEMA
            },
        },
    }
# -----------------------
# Manifest helpers
# -----------------------

def load_manifest() -> Dict[str, Any]:
    """Load the run manifest from disk.

    Args:
        None

    Returns:
        dict: Manifest dictionary if present; otherwise an empty dict.
    """
    mf = WORK_DIR / "manifest.json"
    if mf.exists():
        return json.loads(mf.read_text(encoding="utf-8"))
    return {}

def save_manifest(manifest: Dict[str, Any]) -> None:
    """Write the manifest dictionary to disk as pretty-printed JSON.

    Args:
        manifest (dict): Manifest object to persist.

    Returns:
        None
    """
    mf = WORK_DIR / "manifest.json"
    mf.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

def init_manifest_if_needed() -> Dict[str, Any]:
    """Initialize a new manifest if none exists, otherwise load the existing one.

    Args:
        None

    Returns:
        dict: The current manifest (newly created or loaded).
    """
    ensure_dirs()
    manifest = load_manifest()
    if manifest:
        return manifest

    manifest = {
        "created_utc": utc_now_iso(),
        "model": MODEL,
        "completion_window": COMPLETION_WINDOW,
        "system_rules_sha256": sha256_text(SYSTEM_RULES),
        "schema_sha256": sha256_text(json.dumps(RESPONSE_SCHEMA, sort_keys=True)),
        "few_shot_sha256": sha256_text(json.dumps(few_shot, ensure_ascii=False, sort_keys=True)),
        "miu_json_path": str(MIU_JSON_PATH),
        "few_shot_path": str(FEW_SHOT_PATH),
        "chunks": [],
    }
    save_manifest(manifest)
    return manifest

def chunk_input_paths() -> List[Path]:
    """List chunk input JSONL files on disk in sorted order.

    Args:
        None

    Returns:
        list[pathlib.Path]: Paths to chunk JSONL inputs (e.g., batch_part_0000.jsonl).
    """
    return sorted(INPUT_DIR.glob("batch_part_*.jsonl"))

def _new_chunk_path(i: int) -> Path:
    """Construct the file path for the i-th chunk input JSONL.

    Args:
        i (int): Zero-based chunk index.

    Returns:
        pathlib.Path: Target path for the chunk JSONL file.
    """
    return INPUT_DIR / f"batch_part_{i:04d}.jsonl"

def build_chunks_from_corpus(manifest: Dict[str, Any]) -> Dict[str, Any]:
    """Create deterministic chunk JSONLs from the corpus and register them in the manifest.

    Args:
        manifest (dict): Current manifest to update in-place.

    Returns:
        dict: Updated manifest including chunk entries and metadata.
    """
    ensure_dirs()

    existing = {p.name: p for p in chunk_input_paths()}
    existing_names = set(existing.keys())
    manifest_names = {Path(ch["input_path"]).name for ch in manifest.get("chunks", [])}

    if existing_names and existing_names == manifest_names:
        print(f"Chunks already exist ({len(existing_names)}). Skipping chunk build.")
        return manifest

    if existing_names:
        print("Found chunk files on disk; syncing manifest without rewriting files.")
        for name in sorted(existing_names):
            if name not in manifest_names:
                manifest["chunks"].append({
                    "chunk_name": name.replace(".jsonl", ""),
                    "input_path": str(existing[name]),
                    "output_path": None,
                    "batch_id": None,
                    "batch_id_history": [],
                    "status": "not_submitted",
                    "submitted_utc": None,
                    "completed_utc": None,
                    "attempt": 0,
                    "records": None,
                    "bytes": None,
                    "last_terminal_status": None,
                    "last_error": None,
                    "next_retry_utc": None,
                })
        save_manifest(manifest)
        return manifest

    print("Building chunked JSONL inputs...")
    chunk_idx = 0
    records_in_chunk = 0
    bytes_in_chunk = 0

    out_path = _new_chunk_path(chunk_idx)
    f = out_path.open("w", encoding="utf-8")

    def close_chunk():
        """Finalize the current chunk file handle and update counters.
        """
        nonlocal f, out_path, chunk_idx, records_in_chunk, bytes_in_chunk
        f.close()
        manifest["chunks"].append({
            "chunk_name": out_path.stem,
            "input_path": str(out_path),
            "output_path": None,
            "batch_id": None,
            "batch_id_history": [],
            "status": "not_submitted",
            "submitted_utc": None,
            "completed_utc": None,
            "attempt": 0,
            "records": records_in_chunk,
            "bytes": bytes_in_chunk,
            "last_terminal_status": None,
            "last_error": None,
            "next_retry_utc": None,
        })
        chunk_idx += 1
        records_in_chunk = 0
        bytes_in_chunk = 0

    def open_next_chunk():
        """Open the next chunk JSONL file for streaming writes.
        """
        nonlocal f, out_path
        out_path = _new_chunk_path(chunk_idx)
        return out_path.open("w", encoding="utf-8")

    skipped = 0
    written = 0

    for miu_id, rec in iter_mius_sorted(miu_data):
        miu_text = (rec.get("txt") or "").strip()
        if not miu_text:
            continue
        if SKIP_ALREADY_ANNOTATED and rec.get("ann", {}).get("monetary_info") is not None:
            skipped += 1
            continue

        line_obj = batch_record(miu_id, miu_text)
        line = json.dumps(line_obj, ensure_ascii=False) + "\n"
        line_bytes = len(line.encode("utf-8"))

        if (records_in_chunk >= MAX_RECORDS_PER_CHUNK) or (bytes_in_chunk + line_bytes > MAX_BYTES_PER_CHUNK):
            close_chunk()
            f = open_next_chunk()

        f.write(line)
        records_in_chunk += 1
        bytes_in_chunk += line_bytes
        written += 1

    if records_in_chunk > 0:
        close_chunk()
    else:
        f.close()
        try:
            out_path.unlink()
        except Exception:
            pass

    save_manifest(manifest)
    print(f"Done. Wrote {written:,} MIUs into {len(manifest['chunks'])} chunk(s). Skipped already-annotated: {skipped:,}.")
    return manifest

# -----------------------
# Batch API helpers
# -----------------------

def submit_batch(jsonl_path: Path, completion_window: str = COMPLETION_WINDOW) -> str:
    """Upload a chunk JSONL and create a Batch job, recording the returned batch_id.

    Args:
        jsonl_path (pathlib.Path): Path to the chunk JSONL input file.
        completion_window (str): Batch completion window (e.g., '24h').

    Returns:
        str: Batch job identifier.
    """
    with jsonl_path.open("rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    job = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/responses",
        completion_window=completion_window,
    )
    return job.id

def retrieve_batch(batch_id: str) -> Any:
    """Retrieve the latest Batch job object from the API.

    Args:
        batch_id (str): Batch job identifier.

    Returns:
        Any: Batch object as returned by the OpenAI client.
    """
    return client.batches.retrieve(batch_id)

def fetch_batch_output(batch_id: str, out_jsonl: Path) -> None:
    """Download a completed Batch output file and save it as JSONL.

    Args:
        batch_id (str): Batch job identifier.
        out_jsonl (pathlib.Path): Destination path for the downloaded output JSONL.

    Returns:
        None
    """
    job = retrieve_batch(batch_id)
    if job.status != "completed":
        raise RuntimeError(f"Batch not completed (status={job.status}).")
    if not job.output_file_id:
        raise RuntimeError("No output_file_id found on the completed batch job.")
    text = client.files.content(job.output_file_id).text
    out_jsonl.write_text(text, encoding="utf-8")

# -----------------------
# Retry logic
# -----------------------

def _seconds_until_retry(attempt: int) -> int:
    """Compute exponential backoff delay (with jitter) for retry attempts.

    Args:
        attempt (int): 1-based retry attempt number.

    Returns:
        int: Seconds to wait before the next retry, capped by configuration.
    """
    base = RETRY_BACKOFF_BASE_SECONDS * (2 ** max(0, attempt - 1))
    jitter = random.randint(0, RETRY_BACKOFF_BASE_SECONDS // 2 if RETRY_BACKOFF_BASE_SECONDS >= 2 else 1)
    return min(RETRY_BACKOFF_MAX_SECONDS, int(base + jitter))

def mark_terminal_status(manifest: Dict[str, Any], chunk: Dict[str, Any], terminal_status: str, job_obj: Any) -> None:
    """Update a chunk entry with a terminal job status and any error details.

    Args:
        manifest (dict): Manifest object to update.
        chunk (dict): Chunk entry within the manifest.
        terminal_status (str): Terminal status string (e.g., 'completed', 'failed').
        job_obj (Any): Batch job object returned by the API.

    Returns:
        None
    """
    chunk["last_terminal_status"] = terminal_status
    # Some SDK versions expose error info differently; store a best-effort dump.
    try:
        chunk["last_error"] = job_obj.model_dump()
    except Exception:
        chunk["last_error"] = {"status": getattr(job_obj, "status", None)}

def eligible_for_retry(chunk: Dict[str, Any]) -> bool:
    """Determine whether a chunk is eligible for resubmission.

    Args:
        chunk (dict): Chunk entry from the manifest.

    Returns:
        bool: True if the chunk should be retried; otherwise False.
    """
    status = chunk.get("status")
    if status not in RETRY_ON_STATUSES:
        return False
    attempt = int(chunk.get("attempt") or 0)
    if attempt >= MAX_RETRY_ATTEMPTS:
        return False
    # If a next_retry_utc exists, don't retry until that time (best-effort; lexical compare ok for UTC ISO).
    nrt = chunk.get("next_retry_utc")
    if nrt and utc_now_iso() < nrt:
        return False
    return True

def retry_failed_chunks(manifest: Dict[str, Any]) -> Tuple[Dict[str, Any], int]:
    """Resubmit chunks that failed/cancelled/expired, respecting max attempts and backoff.

    Args:
        manifest (dict): Current manifest to update.

    Returns:
        tuple[dict, int]: (updated_manifest, number_of_retries_submitted).
    """
    retried = 0

    for ch in manifest.get("chunks", []):
        if not eligible_for_retry(ch):
            continue

        input_path = Path(ch["input_path"])
        if not input_path.exists():
            ch["last_error"] = {"kind": "missing_input_path", "path": str(input_path)}
            continue

        # Move previous batch_id to history
        prev = ch.get("batch_id")
        if prev:
            ch.setdefault("batch_id_history", []).append(prev)

        # Submit new attempt
        new_batch_id = submit_batch(input_path, completion_window=COMPLETION_WINDOW)
        ch["batch_id"] = new_batch_id
        ch["attempt"] = int(ch.get("attempt") or 0) + 1
        ch["status"] = "submitted"
        ch["submitted_utc"] = utc_now_iso()
        ch["completed_utc"] = None
        ch["output_path"] = None

        # Backoff window applies if this attempt fails again; set next retry time now
        wait_seconds = _seconds_until_retry(ch["attempt"])
        # Store the earliest retry time as a hint; user can still force by clearing the field.
        ch["next_retry_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(time.time() + wait_seconds))

        retried += 1
        print(f"Retried {ch['chunk_name']} -> {new_batch_id} (attempt {ch['attempt']}/{MAX_RETRY_ATTEMPTS})")

    if retried:
        save_manifest(manifest)

    return manifest, retried

# -----------------------
# Parsing utilities (robustly find JSON inside response bodies)
# -----------------------

_JSON_RE = re.compile(r"^\s*(\{.*\}|\[.*\])\s*$", re.DOTALL)

def _find_first_json(obj: Any) -> Optional[Any]:
    """Find and return the first JSON object/array embedded within a nested structure.

    Args:
        obj (Any): Nested dict/list/string container to search.

    Returns:
        Any | None: Parsed JSON object/array if found, else None.
    """
    if isinstance(obj, str):
        s = obj.strip()
        if _JSON_RE.match(s):
            try:
                return json.loads(s)
            except Exception:
                return None
        return None
    if isinstance(obj, dict):
        for v in obj.values():
            found = _find_first_json(v)
            if found is not None:
                return found
    if isinstance(obj, list):
        for v in obj:
            found = _find_first_json(v)
            if found is not None:
                return found
    return None

PLAINTEXT_SCHEMA = RESPONSE_SCHEMA["schema"]

@dataclass
class ParsedBatch:
    """Parsed results + errors keyed by miu_id."""
    results_by_id: Dict[str, Dict[str, Any]]
    errors_by_id: Dict[str, Dict[str, Any]]

def iter_jsonl(path: Path) -> Iterable[Tuple[int, str]]:
    """Stream lines from a JSONL file without loading it into memory.

    Args:
        path (pathlib.Path): Path to a UTF-8 JSONL file.

    Yields:
        tuple[int, str]: (line_number, raw_line) pairs.
    """
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip("\n")
            if line:
                yield i, line

def parse_output_line(obj: Dict[str, Any]) -> Tuple[Optional[str], Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    """Parse one Batch output JSONL object into (miu_id, parsed_json, error_info).

    Args:
        obj (dict): One JSONL record from the Batch output file.

    Returns:
        tuple[str | None, dict | None, dict | None]: (miu_id, result_json, error_dict).
    """
    custom_id = obj.get("custom_id")
    if not custom_id:
        return None, None, {"kind": "missing_custom_id", "obj": obj}

    if obj.get("error"):
        return custom_id, None, {"kind": "api_error", "error": obj["error"]}

    resp = (obj.get("response") or {})
    status_code = resp.get("status_code")
    body = resp.get("body")

    if status_code != 200:
        return custom_id, None, {"kind": "non_200", "status_code": status_code, "body": body}

    parsed = _find_first_json(body)
    if parsed is None:
        return custom_id, None, {"kind": "no_json_found", "body": body}

    try:
        jsonschema_validate(instance=parsed, schema=PLAINTEXT_SCHEMA)
    except ValidationError as ve:
        return custom_id, None, {"kind": "schema_validation_failed", "error": str(ve), "parsed": parsed}

    return custom_id, parsed, None

def parse_output_jsonl(path: Path) -> ParsedBatch:
    """Parse a per-chunk output JSONL file into a structured in-memory object.

    Args:
        path (pathlib.Path): Path to a per-chunk output JSONL file.

    Returns:
        ParsedBatch: Parsed results and errors for that chunk.
    """
    results_by_id: Dict[str, Dict[str, Any]] = {}
    errors_by_id: Dict[str, Dict[str, Any]] = {}

    for line_no, line in iter_jsonl(path):
        try:
            obj = json.loads(line)
        except Exception as e:
            errors_by_id[f"__line_{line_no}__"] = {"kind": "bad_jsonl_line", "error": str(e), "line": line[:500]}
            continue

        miu_id, parsed, err = parse_output_line(obj)
        if miu_id is None:
            errors_by_id[f"__line_{line_no}__"] = err or {"kind": "unknown_parse_error"}
            continue
        if err:
            errors_by_id[miu_id] = err
        else:
            results_by_id[miu_id] = parsed

    return ParsedBatch(results_by_id, errors_by_id)

# -----------------------
# Merge utilities
# -----------------------

def merge_one_result_into_mius(
    miu_data: Dict[str, Dict[str, Any]],
    miu_id: str,
    result: Dict[str, Any],
    require_nonempty: bool = True,
) -> bool:
    """Merge one validated extraction result into the in-memory MIU corpus structure.

    Args:
        miu_data (dict): Corpus mapping miu_id to MIU record (modified in-place).
        miu_id (str): MIU identifier to merge into.
        result (dict): Validated monetary_info JSON object.
        errors_by_id (dict): Error accumulator mapping miu_id to error info.

    Returns:
        None
    """
    monetary_info = result.get("monetary_info", [])
    if require_nonempty and (not monetary_info):
        return False

    rec = miu_data.get(miu_id) or {}
    ann = dict(rec.get("ann", {}))
    ann["monetary_info"] = monetary_info
    rec = dict(rec)
    rec["ann"] = ann
    miu_data[miu_id] = rec
    return True

def stream_merge_outputs_into_mius(
    miu_data: Dict[str, Dict[str, Any]],
    output_paths: List[Path],
    require_nonempty: bool = True,
) -> Tuple[Dict[str, Any], List[str], Dict[str, Any]]:
    """Streaming parse and merge of many per-chunk outputs into the corpus (memory-efficient).

    Args:
        miu_data (dict): Corpus mapping miu_id to MIU record (modified in-place).
        output_paths (list[pathlib.Path]): Per-chunk output JSONL files to merge.
        schema (dict): JSON Schema used to validate extracted results.

    Returns:
        dict: Summary statistics (counts merged, errors, skipped).
    """
    errors_by_id: Dict[str, Any] = {}
    empty_hits: List[str] = []
    stats = {"lines": 0, "merged": 0, "empty": 0, "errors": 0}

    for p in output_paths:
        for line_no, line in iter_jsonl(p):
            stats["lines"] += 1
            try:
                obj = json.loads(line)
            except Exception as e:
                stats["errors"] += 1
                errors_by_id[f"{p.name}::__line_{line_no}__"] = {"kind": "bad_jsonl_line", "error": str(e), "line": line[:500]}
                continue

            miu_id, parsed, err = parse_output_line(obj)
            if miu_id is None:
                stats["errors"] += 1
                errors_by_id[f"{p.name}::__line_{line_no}__"] = err or {"kind": "unknown_parse_error"}
                continue

            if err:
                stats["errors"] += 1
                errors_by_id[miu_id] = {"file": p.name, **err}
                continue

            merged = merge_one_result_into_mius(miu_data, miu_id, parsed, require_nonempty=require_nonempty)
            if merged:
                stats["merged"] += 1
            else:
                stats["empty"] += 1
                empty_hits.append(miu_id)

    return errors_by_id, empty_hits, stats


In [ ]:
# -----------------------
# Step 1: Build chunks + manifest
# -----------------------
manifest = init_manifest_if_needed()
manifest = build_chunks_from_corpus(manifest)

print("Manifest path:", WORK_DIR / "manifest.json")
print("Chunks in manifest:", len(manifest.get("chunks", [])))


Building chunked JSONL inputs...
Done. Wrote 10 MIUs into 2 chunk(s). Skipped already-annotated: 0.
Manifest path: batch_work/manifest.json
Chunks in manifest: 2


In [ ]:
# -----------------------
# Step 2: Submit pending chunks
# -----------------------
manifest = load_manifest()
changed = False

for ch in manifest.get("chunks", []):
    if ch.get("status") != "not_submitted":
        continue
    input_path = Path(ch["input_path"])
    if not input_path.exists():
        raise FileNotFoundError(f"Missing input chunk file: {input_path}")

    print(f"Submitting {input_path.name} ...")
    batch_id = submit_batch(input_path, completion_window=COMPLETION_WINDOW)
    ch["batch_id"] = batch_id
    ch["status"] = "submitted"
    ch["submitted_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    changed = True
    print("  batch_id =", batch_id)

if changed:
    save_manifest(manifest)
    print("Updated manifest with submitted batch_ids.")
else:
    print("No chunks needed submission.")


Submitting batch_part_0000.jsonl ...
  batch_id = batch_69989b4e7bcc81909af845076f30826e
Submitting batch_part_0001.jsonl ...
  batch_id = batch_69989b4f62e88190adf15325b770c721
Updated manifest with submitted batch_ids.


In [ ]:
# -----------------------
# Step 3: Poll statuses (safe to re-run)
# -----------------------
manifest = load_manifest()
changed = False

for ch in manifest.get("chunks", []):
    batch_id = ch.get("batch_id")
    if not batch_id:
        continue
    if ch.get("status") in ("completed", "failed", "cancelled", "expired"):
        continue

    job = retrieve_batch(batch_id)
    status = job.status
    print(f"{ch['chunk_name']}: {status}")

    # Persist status transitions
    if status != ch.get("status"):
        ch["status"] = status
        changed = True
        if status == "completed":
            ch["completed_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

if changed:
    save_manifest(manifest)
    print("Manifest updated.")
else:
    print("No status changes.")


batch_part_0000: completed
batch_part_0001: completed
Manifest updated.


In [ ]:
# -----------------------
# Step 3b: Automatic retries for failed/cancelled/expired chunks
# -----------------------
manifest = load_manifest()

# Refresh terminal statuses from API (best-effort)
changed = False
for ch in manifest.get('chunks', []):
    batch_id = ch.get('batch_id')
    if not batch_id:
        continue
    if ch.get('status') not in RETRY_ON_STATUSES:
        continue
    try:
        job = retrieve_batch(batch_id)
        # Keep whatever the API reports as the truth
        if job.status != ch.get('status'):
            ch['status'] = job.status
            changed = True
        if job.status in RETRY_ON_STATUSES:
            mark_terminal_status(manifest, ch, job.status, job)
            changed = True
    except Exception as e:
        ch['last_error'] = {'kind': 'retrieve_failed', 'error': str(e)}
        changed = True

if changed:
    save_manifest(manifest)

manifest, retried = retry_failed_chunks(manifest)
print('Retried chunks:', retried)


Retried chunks: 0


In [ ]:
# -----------------------
# Step 4: Download outputs for completed chunks (safe to re-run)
# -----------------------
manifest = load_manifest()
changed = False

for ch in manifest.get("chunks", []):
    if ch.get("status") != "completed":
        continue

    out_path = ch.get("output_path")
    if out_path and Path(out_path).exists():
        continue

    batch_id = ch.get("batch_id")
    if not batch_id:
        continue

    target = OUTPUT_DIR / f"{ch['chunk_name']}.out.jsonl"
    print(f"Downloading output for {ch['chunk_name']} -> {target.name}")
    fetch_batch_output(batch_id, target)
    ch["output_path"] = str(target)
    changed = True

if changed:
    save_manifest(manifest)
    print("Manifest updated with output paths.")
else:
    print("No outputs needed downloading.")


Manifest updated with output paths.


In [ ]:
# -----------------------
# Step 5: Parse outputs + merge into MIU JSON
# -----------------------
manifest = load_manifest()

completed = [
    ch for ch in manifest.get("chunks", [])
    if ch.get("status") == "completed" and ch.get("output_path") and Path(ch["output_path"]).exists()
]
print(f"Completed chunks with outputs: {len(completed)}")

output_paths = [Path(ch["output_path"]) for ch in completed]

# Streaming merge (recommended for 100k+ MIUs)
errors_by_id = {}
empty_hits = []
stats = {}

if STREAMING_MERGE:
    errors_by_id, empty_hits, stats = stream_merge_outputs_into_mius(
        miu_data,
        output_paths,
        require_nonempty=REQUIRE_NONEMPTY,
    )
else:
    # In-memory mode (fine for smaller corpora)
    all_results: Dict[str, Dict[str, Any]] = {}
    all_errors: Dict[str, Dict[str, Any]] = {}
    for ch in completed:
        out_path = Path(ch["output_path"])
        parsed = parse_output_jsonl(out_path)
        all_results.update(parsed.results_by_id)
        for k, v in parsed.errors_by_id.items():
            all_errors[k] = {"chunk": ch["chunk_name"], **v}
    parsed_all = ParsedBatch(results_by_id=all_results, errors_by_id=all_errors)
    # Apply merges
    for miu_id, result in parsed_all.results_by_id.items():
        merged = merge_one_result_into_mius(miu_data, miu_id, result, require_nonempty=REQUIRE_NONEMPTY)
        if not merged:
            empty_hits.append(miu_id)
    errors_by_id = parsed_all.errors_by_id
    stats = {"lines": len(all_results) + len(all_errors), "merged": len(all_results), "empty": len(empty_hits), "errors": len(all_errors)}

# Write artifacts
WORK_DIR.mkdir(parents=True, exist_ok=True)

merged_path = WORK_DIR / "merged_miu.json"
merged_path.write_text(json.dumps(miu_data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", merged_path)

errors_path = WORK_DIR / "errors.json"
errors_path.write_text(json.dumps(errors_by_id, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", errors_path, f"(errors: {len(errors_by_id):,})")

empty_path = WORK_DIR / "empty_hits.json"
empty_path.write_text(json.dumps(empty_hits, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", empty_path, f"(empty hits: {len(empty_hits):,})")

stats_path = WORK_DIR / "merge_stats.json"
stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", stats_path)

print("Stats:", stats)


Completed chunks with outputs: 2
Wrote: batch_work/merged_miu.json
Wrote: batch_work/errors.json (errors: 0)
Wrote: batch_work/empty_hits.json (empty hits: 1)
Wrote: batch_work/merge_stats.json
Stats: {'lines': 10, 'merged': 9, 'empty': 1, 'errors': 0}
